In [11]:
import duckdb


In [12]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [14]:
df = con.execute("SELECT * FROM bronze_produtos").fetch_df()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-11-24 10:11:48.292434
1,10002,MARTELO,BT10,100,1500,z0019_1.csv,2026-11-24 10:11:48.292434
2,10003,PREGO,BT10,100,50,z0019_1.csv,2026-11-24 10:11:48.292434
3,10004,SERRA,BT50,100,200,z0019_2.csv,2026-11-24 10:12:01.645424
4,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-11-24 10:12:01.645424
5,10003,PREGO,BT10,100,60,z0019_2.csv,2026-11-24 10:12:01.645424


In [22]:
df = con.execute("""
SELECT * FROM
(SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) AS row

 FROM bronze_produtos
 WHERE data_ingestao >= '2026-11-19')
 WHERE row = 1
 ORDER BY NATBR
 """).fetch_df()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-11-24 10:11:48.292434,1
1,10002,MARTELO,BT10,100,1500,z0019_1.csv,2026-11-24 10:11:48.292434,1
2,10003,PREGO,BT10,100,60,z0019_2.csv,2026-11-24 10:12:01.645424,1
3,10004,SERRA,BT50,100,200,z0019_2.csv,2026-11-24 10:12:01.645424,1
4,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-11-24 10:12:01.645424,1


In [23]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'row'])
df_final.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT10,100,1500
2,10003,PREGO,BT10,100,60
3,10004,SERRA,BT50,100,200
4,10005,MACHADO,BT50,100,100


In [24]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'row'])
df_final = df_final.rename(columns={"NATBR": "ID"})
df_final = df_final.rename(columns={"MAKTX": "NOME"})
df_final = df_final.rename(columns={"WERKS": "CATEGORIA"})
df_final = df_final.rename(columns={"MAINS": "FORNECEDOR"})
df_final = df_final.rename(columns={"LABST": "PREÇO"})
df_final.head(10)

,ID,NOME,CATEGORIA,FORNECEDOR,PREÇO
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT10,100,1500
2,10003,PREGO,BT10,100,60
3,10004,SERRA,BT50,100,200
4,10005,MACHADO,BT50,100,100


In [25]:
df_final.dtypes

ID            str
NOME          str
CATEGORIA     str
FORNECEDOR    str
PREÇO         str
dtype: object

In [26]:
df2 = df_final
df2 = df2.astype(
{'ID': 'Int32',
 'NOME': 'str',
 'CATEGORIA': 'str',
 'FORNECEDOR': 'Int32',
 'PREÇO': 'Float32'}
)
df2.head(10)

,ID,NOME,CATEGORIA,FORNECEDOR,PREÇO
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT10,100,1500.0
2,10003,PREGO,BT10,100,60.0
3,10004,SERRA,BT50,100,200.0
4,10005,MACHADO,BT50,100,100.0


In [27]:
df2.dtypes

ID              Int32
NOME              str
CATEGORIA         str
FORNECEDOR      Int32
PREÇO         Float32
dtype: object

In [28]:
con.execute("""
CREATE TABLE IF NOT EXISTS produtos (
ID BIGINT,
NOME TEXT,
CATEGORIA TEXT,
FORNECEDOR BIGINT,
PREÇO FLOAT,
)
""")

In [29]:
df2.head(10)

,ID,NOME,CATEGORIA,FORNECEDOR,PREÇO
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT10,100,1500.0
2,10003,PREGO,BT10,100,60.0
3,10004,SERRA,BT50,100,200.0
4,10005,MACHADO,BT50,100,100.0


In [30]:
df_resultado = con.execute("select * from produtos").fetch_df()
df_resultado.head(10)

,ID,NOME,CATEGORIA,FORNECEDOR,PREÇO


In [31]:
df2.head(10)

,ID,NOME,CATEGORIA,FORNECEDOR,PREÇO
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT10,100,1500.0
2,10003,PREGO,BT10,100,60.0
3,10004,SERRA,BT50,100,200.0
4,10005,MACHADO,BT50,100,100.0


In [32]:
con.execute("INSERT INTO produtos SELECT *FROM df2")

In [33]:
df_resultado = con.execute("select * from produtos").fetch_df()
df_resultado.head(10)

,ID,NOME,CATEGORIA,FORNECEDOR,PREÇO
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT10,100,1500.0
2,10003,PREGO,BT10,100,60.0
3,10004,SERRA,BT50,100,200.0
4,10005,MACHADO,BT50,100,100.0


In [34]:
con.close()